# Credit card fraud detection

## Introduction

The machine learning project aims to detect fraudulent credit card transaction, which is a growing problem for the banking industry and also for consumers. Due to growing scam and social engineering activities more people are victims. Therefore there are multiple stakeholders (regulatory authorities EBA & ECB, banks, consumers) interested in a sufficient fraud prevention and detection. 



This project considers the current developments in academic discussion and uses their approaches, mainly the recent published paper by "Shi et al (2025): An attention-based balanced variational autoencoder method for credit card fraud detection", on a different dataset which is publicly available on Hugging Face by the following link:

https://huggingface.co/datasets/pointe77/credit-card-transaction

The credit card data (most likely synthetic data) downloaded from Hugging face, without providing any descriptions.
Nevertheless the used variables are quit obvious, by their structure and their content. 
So the qualitative data description will be conducted on industrial experience and best guesses.


In [14]:
# Importing standard packages
import polars as pl # Faster then pandas, scikit-learn native usage since 1.4
import numpy as np
import matplotlib.pyplot as plt # visualization
#import sklearn as 

from geopy.distance import distance, geodesic, great_circle # Feature Engineering module for Geodata

In [6]:
# reading in the datasets
# Read CSV files with Polars
cc_train = pl.read_csv("/home/sarima/Desktop/UNI Olsztyn /5# Machine Learning/ML_Project/credit_card_transaction_train.csv")
cc_test  = pl.read_csv("/home/sarima/Desktop/UNI Olsztyn /5# Machine Learning/ML_Project/credit_card_transaction_test.csv")
cc = pl.concat([cc_train, cc_test])
del(cc_train, cc_test)

Additionally there is taken fraud data from the FBI fillings published via the pdf documentation.
State  data from the FBI.pdf, p.29
Losses per 100,000 citizens in each state

In [7]:
crime_data = pl.DataFrame({
    'State': ['District of Columbia', 'Nevada', 'California', 'New Jersey', 'Arizona', 
              'Alaska', 'Montana', 'South Dakota', 'Utah', 'Florida', 'New York', 
              'Washington', 'Hawaii', 'Maryland', 'Delaware', 'Minnesota', 'Massachusetts', 
              'Texas', 'Connecticut', 'Oregon', 'Kansas', 'Colorado', 'Virginia', 
              'Rhode Island', 'Pennsylvania', 'Georgia', 'Illinois', 'Idaho', 'Indiana', 
              'Wyoming', 'Tennessee', 'South Carolina', 'North Carolina', 'New Mexico', 
              'Nebraska', 'Michigan', 'Missouri', 'New Hampshire', 'Alabama', 'Iowa', 
              'North Dakota', 'Louisiana', 'Ohio', 'Oklahoma', 'Wisconsin', 'Arkansas', 
              'Puerto Rico', 'Vermont', 'Maine', 'West Virginia', 'Mississippi', 'Kentucky'],

    'State_ID': [ 'DC',    'NV',     'CA',     'NJ',     'AZ',
     'AK',     'MT',     'SD',     'UT',     'FL',     'NY',
     'WA',     'HI',     'MD',     'DE',     'MN',     'MA',
     'TX',     'CT',     'OR',     'KS',     'CO',     'VA',
     'RI',    'PA',     'GA',     'IL',     'ID',     'IN',
     'WY',     'TN',     'SC',     'NC',     'NM',     'NE',
     'MI',     'MO',     'NH',    'AL',     'IA',     'ND',
     'LA',     'OH', 'OK',     'WI',     'AR',     'PR',
     'VT',     'ME',     'WV',     'MS',     'KY'],
     
    'Loss in $/Capita': [6795914, 6292550, 5542009, 4748238, 4364657, 4332018, 4021353, 3900228,
             3869729, 3868631, 3831931, 3695066, 3603978, 3584328, 3428347, 3380137,
             3369186, 3348973, 3338719, 3213809, 3202070, 3192143, 3041335, 2882110,
             2779999, 2729130, 2675478, 2577030, 2364534, 2353556, 2261914, 2232240,
             2168543, 2134317, 2051237, 2026907, 1991645, 1938461, 1888622, 1865588,
             1726240, 1711639, 1674584, 1651948, 1557861, 1518551, 1479384, 1361957,
             1359051, 1211587, 1093451, 1076986]
})

Even though this specific dataset seems to be synthetically derived and the names and personal data is most likelly not critical to process, in the next steps there will be an anomisation of personel data for the following reasons:

- Depending on the contract between the bank and a third party consulting firm, the data might be depriaciated by the personal information. Therefore there is no simple deletion of the First and Last name of the data, but by an hashing algorithm an anomisation but still conserving client data informations of multiple credit cards. In reality it seems to be a realistic scenario that a victim loosing his credit card information, by social engineering or different scamming attacs might also loose multiple credit card information and therefore there might be a correlation with a causal background.

- The dataset contains also data for gender and date of birth, which is from an ethical viewpoint critical by a following discrimination. The results of the algorithm could suggest that a specific gender or age is more vulnerable to scam and therefore could be excluded from banking business or get an risk premium, by simply being in that socio economic group. In real application this should be considered.  

In [8]:
# hashing name data for anominization and deleting prior columns
cc = cc.with_columns(
    (pl.col("first").cast(pl.Utf8) + "|" + pl.col("last").cast(pl.Utf8)).alias("Owner_ID")
)

cc = cc.with_columns(
    pl.col("Owner_ID").hash().cast(pl.UInt64).alias("Owner_ID")
)

cc = cc.drop(['first', 'last']) 

## First Overview of the Data

### Qualitative Description of the DATA, What the Variables mean.



## Feature Engineering

Since some feature should be considered together, as input into the model are the engineered features instead of the original variables used.

Considering longitude and latitude in combination (both vor the victim, as well as for the merchant) results in information about the distance, whereas only each variable on their own is not sufficent.

Furthermore it is also reducing the dimensionality, by using less input variables within the model.

I.1) Engineered Feature 
Constructing a Risk Rating of each state by using FBI Data on Fraud occurence. If there are states in which due to governmental reasons, like regulation or technical infrasturcture. This feature might be an exogenous variable on our fraud flag. 

In [9]:
# combining States (governmental reasons) and their fraud losses/per Capita (extra dataset) as Risk-Rating Variable
# State_Risk_Rating

crime = crime_data.select([
    pl.col("State_ID"),
    pl.col("Loss in $/Capita").alias("State_Risk_Rating")
])

cc = cc.join(
    crime,
    left_on="state",
    right_on="State_ID",
    how="left"
)


I.2) Engineered Feature
Using the geolocation data, to check if the adress of the client/victim and the merchant is distant. It might indicate either travelling or online shopping, but also suspicous behaviour. An card usage abroad in south east asia, which did not happen before could be at least suspicous, contacting a client and asking for current travel could be a safety feature.


In [10]:
# Using client and merchant distance, rowwise
# from geopy.distance import distance 

cc = cc.with_columns(
    pl.struct(['lat', 'long', 'merch_lat', 'merch_long'])
    .map_elements(
        lambda row: distance((row['lat'], row['long']), 
                            (row['merch_lat'], row['merch_long'])).km,
        return_dtype=pl.Float64
    )
    .alias('dist_client_merchant')
)

I.3) Engineered Feature
This feature is measering the distance of the previous card transaction to the current card transaction in relation to time. Even though due to travel or online shopping, this information does not tell for sure any fraudulent behaviour, but it shows some unusual behaviour, which is at least an indicator. In comparison to the second feature it is a more complex, but also more causal.

In [12]:
# transforming str dtype int dateformat
cc = cc.with_columns(
    pl.col('trans_date_trans_time').str.to_datetime().alias('trans_date_trans_time')
)

In [ ]:
# Using merchant distance and time between transactions for each card
cc = cc.sort(['cc_num', 'trans_date_trans_time'])

# Create lagged values for each card
cc = cc.with_columns([
    pl.col('merch_lat').shift(1).over('cc_num').alias('prev_merch_lat'),
    pl.col('merch_long').shift(1).over('cc_num').alias('prev_merch_long'),
    pl.col('trans_date_trans_time').shift(1).over('cc_num').alias('prev_trans_time')
])

# distance in km, per transaction
cc = cc.with_columns(
    pl.struct(['merch_lat', 'merch_long', 'prev_merch_lat', 'prev_merch_long'])
    .map_elements(
        lambda row: distance(
            (row['merch_lat'], row['merch_long']),
            (row['prev_merch_lat'], row['prev_merch_long'])
        ).km if row['prev_merch_lat'] is not None else 0.0, # None problem with lagged data
    ).alias('dist_prev_current_trans') # polars got a  problem with integer and Float64, therefore 0.0!
)


# transaction time difference
# maybe engineered feature on its own?
cc = cc.with_columns(
    pl.struct(['trans_date_trans_time', 'prev_trans_time'])
    .map_elements(
        lambda row: (row['trans_date_trans_time'] - row['prev_trans_time']).total_seconds()
        if row['prev_trans_time'] is not None else 0.0, # None problem with lagged data
    ).alias('trans_time_diff')
)

# TODO 
# travel time distance -> If physical card usage, great feature
cc = cc.with_columns(
    (pl.col('dist_prev_current_trans') / pl.col('trans_time_diff'))
    .map_elements(lambda x: x if x > 0 else 0.0)
    .alias('travel_time_km')
) # problem with infinity and NAN's 
# Unit Kilometers per second (km/s) (because more precise), per hour is unrealistic if 

(Dornadula & Geetha, 2019): "Firstly, we use clustering method to divide the cardholders into different clusters/groups based on their transaction amount, i.e., high, medium and low using range partitioning.  Using Sliding-Window method, we aggregate the transactions into respective groups, i.e., extract some features from window to find cardholder's behavioural patterns. Features like maximum amount, minimum amount of transaction, followed by the average amount in the window and even the time elapsed."
/* features extraction related to amount */  ai1=MAX_AMT(Ti);  ai2=MIN_AMT(Ti);  ai3=AVG_AMT(Ti);  ai4=AMT(Ti);  For j in range i+w-1:  /* Time elapse */  xi= Time(tj)-Time(tj-1)  End  Xi= (ai1, ai2,ai3,ai4,ai5,);

In [ ]:
# Variable construction, Max amount per client in relation to IQR, but in avoidance to Multicollinearity,
# -> transformation in Outlier binary Variable. Flag unusual Spending Transaction.



In [ ]:
def stat_values(df: pl.DataFrame, numeric_vars: list) -> pl.DataFrame:
    
    stats_list = []
    for col in numeric_vars: 
        stats = {
            "variable": col,
            "mean": df[col].mean(),
            "median": df[col].median(),
            "std": df[col].std(),
            "variance": df[col].var(),
            "min": df[col].min(),
            "q1": df[col].quantile(0.25),
            "q3": df[col].quantile(0.75),
            "max": df[col].max(),
            "range": df[col].max() - df[col].min(),
            "iqr": df[col].quantile(0.75) - df[col].quantile(0.25),
            "skewness": df[col].skew(),
            "kurtosis": df[col].kurtosis(),
        }
        stats_list.append(stats)
    
    return pl.DataFrame(stats_list)

# Usage
stats_df = stat_values(cc, ['amt','State_Risk_Rating','dist_client_merchant','dist_prev_current_trans','trans_time_diff','travel_time_km'])
print(stats_df)

shape: (6, 13)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ variable  ┆ mean      ┆ median    ┆ std       ┆ … ┆ range     ┆ iqr       ┆ skewness  ┆ kurtosis │
│ ---       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│ str       ┆ f64       ┆ f64       ┆ f64       ┆   ┆ f64       ┆ f64       ┆ f64       ┆ f64      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ amt       ┆ 70.063567 ┆ 47.45     ┆ 159.25397 ┆ … ┆ 28947.9   ┆ 73.46     ┆ 40.812776 ┆ 4181.896 │
│           ┆           ┆           ┆ 5         ┆   ┆           ┆           ┆           ┆ 053      │
│ State_Ris ┆ 2.7786e6  ┆ 2.72913e6 ┆ 1.1181e6  ┆ … ┆ 5.718928e ┆ 1.491515e ┆ 0.756577  ┆ 0.394451 │
│ k_Rating  ┆           ┆           ┆           ┆   ┆ 6         ┆ 6         ┆           ┆          │
│ dist_clie ┆ 76.109558 ┆ 78.248227 ┆ 29.092731 ┆ … ┆ 151.84592 ┆ 43.130061 

In [18]:
stats_df

variable,mean,median,std,variance,min,q1,q3,max,range,iqr,skewness,kurtosis
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""amt""",70.063567,47.45,159.253975,25361.828481,1.0,9.64,83.1,28948.9,28947.9,73.46,40.812776,4181.896053
"""State_Risk_Rating""",2.7786e6,2.72913e6,1.1181e6,1.2500e12,1.076986e6,1.888622e6,3.380137e6,6.795914e6,5.718928e6,1.491515e6,0.756577,0.394451
"""dist_client_merchant""",76.109558,78.248227,29.092731,846.387002,0.022274,55.341983,98.472044,151.8682,151.845927,43.130061,-0.237738,-0.632008
"""dist_prev_current_trans""",103.524514,100.74801,50.412919,2541.462397,0.0,64.338283,139.684519,290.00496,290.00496,75.346236,0.252468,-0.60015
"""trans_time_diff""",30922.794614,15680.0,45322.458627,2.0541e9,0.0,5686.0,38115.0,1.341471e6,1.341471e6,32429.0,4.299994,32.448034
"""travel_time_km""",inf,0.00586,NaN,NaN,0.0,0.002215,0.017238,inf,inf,0.015023,NaN,NaN


Amounts got extreme Outliers. In Basic Data.
Transformation (No 0 or negative Values makes it easier) -> log transformation.

In [21]:
def categorical_stats(df: pl.DataFrame, categorical_vars: list) -> pl.DataFrame:
    stats_list = []
    for col in categorical_vars:
        
        value_counts = df[col].value_counts()
                
        most_frequent = value_counts[0, col]
        most_freq_count = value_counts[0, "count"] 
        
        stats = {
            "variable": col,
            "n_unique": df[col].n_unique(),
            "most_frequent": most_frequent,
            "most_freq_pct": (most_freq_count / df.height) * 100,
        }
        stats_list.append(stats)
    
    return pl.DataFrame(stats_list)

cat_stats = categorical_stats(cc, ['category','gender','state','job','merchant'])


In [22]:
cat_stats

variable,n_unique,most_frequent,most_freq_pct
str,i64,str,f64
"""category""",14,"""entertainment""",7.240252
"""gender""",2,"""F""",54.780408
"""state""",51,"""KY""",2.212326
"""job""",497,"""Financial trader""",0.434789
"""merchant""",693,"""fraud_Wuckert, Wintheiser and …",0.188621


II.1) Feature Representation
TODO:
By using a ordinal range/categories for the variables,  it results in some better model performances. 
(Linear SVM, Logistic Regression) -Linear SVM for this data not used.
Whereas we will use the scaled integer values, for our tree based models. (XGBoost, LightGBM)  

The following Variables are therefore transformed:

- City Population
- State Risk Rating

Ordinal text variables:
Following the Lab4 - OrdinalEncoder will be used from the preprocessing module in Scikit-learn. 